# Подготовка полного датасета

1. Загружаем RU и EN датасеты.
2. Загружаем Helsinki-NLP/opus-mt-en-ru и его родной tokenizer.
3. Английские новости разбиваем на предложения.
4. Длинные предложения дополнительно делим на chunks.
5. Переводим каждое предложение/chunk отдельно.
6. Собираем обратно исходные новости.
7. Объединяем RU + EN→RU.
8. Сохраняем `full_dataset.csv`.

In [2]:
from pathlib import Path
import re

import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# =========================
# CONFIG
# =========================

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"

# Фактический лимит модели = 512.
# Оставляем небольшой запас.
MAX_SOURCE_TOKENS = 500

BATCH_SIZE = 32
NUM_BEAMS = 4


# =========================
# PATHS
# =========================

DATA_DIR = Path("../data/prepared_datasets")

RU_PATH = DATA_DIR / "ru_dataset.csv"
EN_PATH = DATA_DIR / "en_dataset.csv"
OUT_PATH = DATA_DIR / "full_dataset.csv"


# =========================
# DEVICE
# =========================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Data:", DATA_DIR.resolve())
print("Output:", OUT_PATH.resolve())
print()
print("PyTorch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Data: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets
Output: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\full_dataset.csv

PyTorch: 2.9.1+rocm7.2.1
ROCm/HIP: 7.2.53211-158bd99533
Device: cuda
GPU: AMD Radeon RX 9070


## 2. Загрузка и проверка датасетов

In [3]:
ru_df = pd.read_csv(RU_PATH)
en_df = pd.read_csv(EN_PATH)


required_ru = {"text", "label", "source"}
required_en = {"text", "label", "source", "orig_lang"}

missing_ru = required_ru - set(ru_df.columns)
missing_en = required_en - set(en_df.columns)

if missing_ru:
    raise ValueError(
        f"В ru_dataset.csv отсутствуют: {missing_ru}"
    )

if missing_en:
    raise ValueError(
        f"В en_dataset.csv отсутствуют: {missing_en}"
    )


print("RU:", ru_df.shape)
print("EN:", en_df.shape)
print()
print("Структура корректна.")

RU: (19797, 3)
EN: (50446, 4)

Структура корректна.


## 3. Загрузка модели и её родного tokenizer

Используется tokenizer именно от `Helsinki-NLP/opus-mt-en-ru`.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

translator = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

translator = translator.to(DEVICE)
translator.eval()


print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print(
    "Max position embeddings:",
    translator.config.max_position_embeddings
)


# =========================
# TEST
# =========================

test_sentences = [
    "Scientists have announced the discovery of a new species of animal in a remote region of South America.",
    "Researchers say the species has never been documented before."
]

inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    padding=True,
    truncation=False
)

inputs = {
    k: v.to(DEVICE)
    for k, v in inputs.items()
}

with torch.inference_mode():
    generated = translator.generate(
        **inputs,
        num_beams=NUM_BEAMS,
        max_new_tokens=128,
        early_stopping=False,
        renormalize_logits=True
    )

test_translation = tokenizer.batch_decode(
    generated,
    skip_special_tokens=True
)

print("\nTEST:")
for src, dst in zip(test_sentences, test_translation):
    print("\nEN:", src)
    print("RU:", dst)

c:\Users\user\miniconda3\envs\main\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model: Helsinki-NLP/opus-mt-en-ru
Device: cuda
Max position embeddings: 512


c:\Users\user\miniconda3\envs\main\Lib\site-packages\transformers\integrations\sdpa_attention.py:158: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TEST:

EN: Scientists have announced the discovery of a new species of animal in a remote region of South America.
RU: Ученые объявили об открытии нового вида животных в отдаленном регионе Южной Америки.

EN: Researchers say the species has never been documented before.
RU: Исследователи говорят, что этот вид никогда раньше не документировался.


## 4. Разбиение текста на предложения и длинные chunks

Обычная новость переводится предложение за предложением.

Если отдельное предложение превышает 500 токенов, оно дополнительно разбивается по токенам.

In [5]:
def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            str(text),
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )


def split_by_tokens(
    text: str,
    max_tokens: int = MAX_SOURCE_TOKENS
) -> list[str]:

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    chunks = []

    for start in range(
        0,
        len(token_ids),
        max_tokens
    ):

        chunk_ids = token_ids[
            start:start + max_tokens
        ]

        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()

        if chunk:
            chunks.append(chunk)

    return chunks


def split_into_sentences(text: str) -> list[str]:
    """
    Простое разбиение новостного текста
    по концам предложений.
    """

    text = str(text).strip()

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?…])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


def prepare_translation_units(
    text: str,
    max_tokens: int = MAX_SOURCE_TOKENS
) -> tuple[list[str], int]:

    sentences = split_into_sentences(text)

    units = []
    chunks_for_news = 0

    for sentence in sentences:

        sentence_tokens = count_tokens(sentence)

        # Обычное предложение
        if sentence_tokens <= max_tokens:
            units.append(sentence)
            continue

        # Очень длинное предложение
        sentence_chunks = split_by_tokens(
            sentence,
            max_tokens
        )

        units.extend(sentence_chunks)

        # Считаем именно дополнительные chunks
        chunks_for_news += len(sentence_chunks)

    return units, chunks_for_news

## 5. Анализ длины английского датасета

Считаем превышение лимита модели и количество новостей, которым потребовался chunking.

In [6]:
en_texts = (
    en_df["text"]
    .fillna("")
    .astype(str)
    .tolist()
)


token_counts = []
translation_units = []
chunk_counts = []


for text in tqdm(
    en_texts,
    desc="Preparing texts"
):

    token_count = count_tokens(text)

    units, long_chunks = prepare_translation_units(
        text,
        MAX_SOURCE_TOKENS
    )

    token_counts.append(token_count)
    translation_units.append(units)

    # Если новость целиком > 500,
    # значит она потребовала разбиения.
    if token_count > 512:
        chunk_counts.append(len(units))
    else:
        chunk_counts.append(1)


token_counts = pd.Series(
    token_counts,
    index=en_df.index
)

chunk_counts = pd.Series(
    chunk_counts,
    index=en_df.index
)


# =========================
# STATISTICS
# =========================

num_over_512 = int(
    (token_counts > 512).sum()
)

num_chunked_news = int(
    (chunk_counts > 1).sum()
)

total_long_chunks = int(
    chunk_counts[chunk_counts > 1].sum()
)

max_chunks_per_news = int(
    chunk_counts.max()
)


print("=" * 60)
print("ENGLISH DATASET STATISTICS")
print("=" * 60)

print(
    f"Всего английских новостей:       {len(en_df):,}"
)

print(
    f"Новостей > 512 токенов:           {num_over_512:,}"
)

print(
    f"Новостей разбито на chunks:      {num_chunked_news:,}"
)

print(
    f"Всего chunks у длинных новостей: {total_long_chunks:,}"
)

print(
    f"Максимум chunks в одной новости: {max_chunks_per_news}"
)

print(
    f"Максимальная длина новости:      {token_counts.max():,}"
)

Preparing texts:   0%|          | 0/50446 [00:00<?, ?it/s]

ENGLISH DATASET STATISTICS
Всего английских новостей:       50,446
Новостей > 512 токенов:           0
Новостей разбито на chunks:      0
Всего chunks у длинных новостей: 0
Максимум chunks в одной новости: 1
Максимальная длина новости:      224


## 6. Перевод предложений/chunks

Каждая переводческая единица обрабатывается отдельно.

In [7]:
def translate_batch(
    texts: list[str],
    batch_size: int = BATCH_SIZE
) -> list[str]:

    results = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Translating EN → RU"
    ):

        batch = texts[
            start:start + batch_size
        ]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=False
        )

        if inputs["input_ids"].shape[1] > 512:
            raise RuntimeError(
                "В batch попал текст длиннее 512 токенов."
            )

        inputs = {
            k: v.to(DEVICE)
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generated = translator.generate(
                **inputs,
                num_beams=NUM_BEAMS,
                max_new_tokens=256,
                early_stopping=False,
                renormalize_logits=True
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

        results.extend(
            text.strip()
            for text in decoded
        )

    return results


# Все translation units → один плоский список
flat_units = [
    unit
    for units in translation_units
    for unit in units
]


print(
    f"Всего translation units: {len(flat_units):,}"
)


translated_units = translate_batch(
    flat_units,
    batch_size=BATCH_SIZE
)


assert len(flat_units) == len(translated_units)

print(
    f"Переведено: {len(translated_units):,}"
)

Всего translation units: 54,307


Translating EN → RU:   0%|          | 0/1698 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Переведено: 54,307


## 7. Сборка переведённых предложений обратно в новости

In [8]:
translated_news = []

position = 0

for units in translation_units:

    n = len(units)

    translated = translated_units[
        position:position + n
    ]

    position += n

    # Собираем исходную новость обратно.
    combined = " ".join(
        text.strip()
        for text in translated
        if text.strip()
    )

    translated_news.append(combined)


assert position == len(translated_units)
assert len(translated_news) == len(en_df)


print(
    f"Собрано новостей: {len(translated_news):,}"
)

Собрано новостей: 50,446


## 8. Объединение RU + EN→RU

In [9]:
# =========================
# Native Russian
# =========================

ru_prepared = pd.DataFrame({
    "text": ru_df["text"]
        .fillna("")
        .astype(str),

    "label": ru_df["label"]
        .astype(int),

    "source": ru_df["source"]
        .astype(str),

    "orig_lang": "ru",

    "orig_text": "-"
})


# =========================
# English → Russian
# =========================

en_prepared = pd.DataFrame({
    "text": translated_news,

    "label": en_df["label"]
        .astype(int)
        .values,

    "source": en_df["source"]
        .astype(str)
        .values,

    "orig_lang": en_df["orig_lang"]
        .fillna("en")
        .astype(str)
        .values,

    "orig_text": en_df["text"]
        .fillna("")
        .astype(str)
        .values
})


# =========================
# Full dataset
# =========================

full_dataset = pd.concat(
    [
        ru_prepared,
        en_prepared
    ],
    ignore_index=True
)


full_dataset = full_dataset[
    [
        "text",
        "label",
        "source",
        "orig_lang",
        "orig_text"
    ]
]


print(
    "Full dataset:",
    full_dataset.shape
)

Full dataset: (70243, 5)


## 9. Проверка и сохранение

In [10]:
# =========================
# VALIDATION
# =========================

assert len(full_dataset) == (
    len(ru_df) + len(en_df)
)

assert (
    full_dataset.columns.tolist()
    ==
    [
        "text",
        "label",
        "source",
        "orig_lang",
        "orig_text"
    ]
)

assert (
    full_dataset[
        full_dataset["orig_lang"] == "ru"
    ]["orig_text"] == "-"
).all()

assert (
    full_dataset[
        full_dataset["orig_lang"] == "en"
    ]["orig_text"] != "-"
).all()


# =========================
# FINAL REPORT
# =========================

print("=" * 60)
print("FINAL REPORT")
print("=" * 60)

print(
    f"Русских новостей:                 "
    f"{len(ru_prepared):,}"
)

print(
    f"Английских новостей:              "
    f"{len(en_prepared):,}"
)

print(
    f"ИТОГО новостей:                   "
    f"{len(full_dataset):,}"
)

print()

print(
    f"EN новостей > 512 токенов:        "
    f"{num_over_512:,}"
)

print(
    f"EN новостей разбито на chunks:    "
    f"{num_chunked_news:,}"
)

print(
    f"Всего chunks у длинных новостей:  "
    f"{total_long_chunks:,}"
)

print(
    f"Максимум chunks в одной новости: "
    f"{max_chunks_per_news}"
)

print()

print("Original languages:")
print(
    full_dataset["orig_lang"]
    .value_counts()
)

print()

print("Labels:")
print(
    full_dataset["label"]
    .value_counts()
)

print()

print("Missing values:")
print(
    full_dataset.isna().sum()
)


# =========================
# SAVE
# =========================

full_dataset.to_csv(
    OUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print()
print(
    "Saved:",
    OUT_PATH.resolve()
)

FINAL REPORT
Русских новостей:                 19,797
Английских новостей:              50,446
ИТОГО новостей:                   70,243

EN новостей > 512 токенов:        0
EN новостей разбито на chunks:    0
Всего chunks у длинных новостей:  0
Максимум chunks в одной новости: 1

Original languages:
orig_lang
en    50446
ru    19797
Name: count, dtype: int64

Labels:
label
0    39762
1    30481
Name: count, dtype: int64

Missing values:
text         0
label        0
source       0
orig_lang    0
orig_text    0
dtype: int64

Saved: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\full_dataset.csv


# Фильтрация политических и религиозных новостей

Используем русский RuBERT NLI-классификатор.

Для каждой новости проверяем две темы:
- политика
- религия

Если ни одна тема не достигает порога, новость получает `none`.

Из исходного датасета удаляем только `politic` и `religion`.

In [11]:
from pathlib import Path
import re

import pandas as pd
import numpy as np
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "cointegrated/rubert-base-cased-nli-twoway"

DATA_DIR = Path("../data/prepared_datasets")

INPUT_PATH = DATA_DIR / "full_dataset.csv"
OUTPUT_PATH = DATA_DIR / "filtered_full_dataset.csv"

# Для NLI используем небольшой запас относительно 512.
MAX_TOKENS = 450

BATCH_SIZE = 64

# Консервативный порог, чтобы не выкидывать
# большое количество нормальных новостей.
TOPIC_THRESHOLD = 0.70


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("Input:", INPUT_PATH.resolve())
print("Output:", OUTPUT_PATH.resolve())
print()
print("PyTorch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Input: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\full_dataset.csv
Output: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\filtered_full_dataset.csv

PyTorch: 2.9.1+rocm7.2.1
ROCm/HIP: 7.2.53211-158bd99533
Device: cuda
GPU: AMD Radeon RX 9070


## 2. Загрузка датасета и модели

In [12]:
# ============================================================
# DATASET
# ============================================================

df = pd.read_csv(INPUT_PATH)

required_columns = {
    "text",
    "label",
    "source",
    "orig_lang",
    "orig_text"
}

missing = required_columns - set(df.columns)

if missing:
    raise ValueError(
        f"Отсутствуют столбцы: {missing}"
    )

print("Dataset shape:", df.shape)


# ============================================================
# MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model = model.to(DEVICE)
model.eval()


print()
print("Model:", MODEL_NAME)
print("Labels:", model.config.id2label)
print(
    "Max length:",
    tokenizer.model_max_length
)

Dataset shape: (70243, 5)


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

c:\Users\user\miniconda3\envs\main\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--cointegrated--rubert-base-cased-nli-twoway. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

...

c:\Users\user\miniconda3\envs\main\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  711MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model: cointegrated/rubert-base-cased-nli-twoway
Labels: {0: 'not_entailment', 1: 'entailment'}
Max length: 512


## 3. Подготовка длинных новостей

Новости разбиваем на предложения.

Если предложение слишком длинное, дополнительно режем его по токенам.

После классификации результаты всех частей объединяются обратно в одну новость.

In [13]:
def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )


def split_by_tokens(
    text: str,
    max_tokens: int = MAX_TOKENS
) -> list[str]:

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    chunks = []

    for start in range(
        0,
        len(token_ids),
        max_tokens
    ):

        ids = token_ids[
            start:start + max_tokens
        ]

        chunk = tokenizer.decode(
            ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        ).strip()

        if chunk:
            chunks.append(chunk)

    return chunks


def split_into_chunks(
    text: str,
    max_tokens: int = MAX_TOKENS
) -> list[str]:

    text = str(text).strip()

    if not text:
        return [""]

    if count_tokens(text) <= max_tokens:
        return [text]

    # Сначала делим по предложениям.
    sentences = re.split(
        r'(?<=[.!?…])\s+',
        text
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    chunks = []
    current = []
    current_tokens = 0

    for sentence in sentences:

        sentence_tokens = count_tokens(sentence)

        # Слишком длинное отдельное предложение.
        if sentence_tokens > max_tokens:

            if current:
                chunks.append(
                    " ".join(current)
                )

                current = []
                current_tokens = 0

            chunks.extend(
                split_by_tokens(
                    sentence,
                    max_tokens
                )
            )

            continue

        # Добавляем предложение в текущий chunk.
        if current_tokens + sentence_tokens <= max_tokens:

            current.append(sentence)
            current_tokens += sentence_tokens

        else:

            chunks.append(
                " ".join(current)
            )

            current = [sentence]
            current_tokens = sentence_tokens

    if current:
        chunks.append(
            " ".join(current)
        )

    return chunks

## 4. Классификация политики и религии

Используем NLI zero-shot:

- `Текст посвящён политике.`
- `Текст посвящён религии.`

Проверяем темы независимо, поэтому существует настоящий класс `none`.

In [14]:
# ============================================================
# FIND ENTAILMENT ID
# ============================================================

label2id = {
    str(label).lower(): int(idx)
    for label, idx
    in model.config.label2id.items()
}

if "entailment" not in label2id:
    raise RuntimeError(
        f"Не найден entailment в label2id: {label2id}"
    )

ENTAILMENT_ID = label2id["entailment"]


# ============================================================
# HYPOTHESES
# ============================================================

HYPOTHESES = {
    "politic": "Текст посвящён политике.",
    "religion": "Текст посвящён религии."
}


# ============================================================
# CLASSIFY BATCH
# ============================================================

def classify_topic_batch(
    texts: list[str],
    hypothesis: str
) -> np.ndarray:

    hypotheses = [
        hypothesis
    ] * len(texts)

    inputs = tokenizer(
        texts,
        hypotheses,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_TOKENS
    )

    inputs = {
        k: v.to(DEVICE)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = model(
            **inputs
        )

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    entailment_scores = probabilities[
        :, ENTAILMENT_ID
    ]

    return entailment_scores.cpu().numpy()


# ============================================================
# PREPARE NEWS CHUNKS
# ============================================================

all_news_chunks = [
    split_into_chunks(text)
    for text in df["text"].fillna("").astype(str)
]


flat_chunks = [
    chunk
    for chunks in all_news_chunks
    for chunk in chunks
]


print(
    "Новости:",
    len(df)
)

print(
    "Chunks:",
    len(flat_chunks)
)


# ============================================================
# CLASSIFY ALL CHUNKS
# ============================================================

topic_scores = {
    "politic": [],
    "religion": []
}


for topic, hypothesis in HYPOTHESES.items():

    scores = []

    for start in tqdm(
        range(
            0,
            len(flat_chunks),
            BATCH_SIZE
        ),
        desc=f"Classifying {topic}"
    ):

        batch = flat_chunks[
            start:start + BATCH_SIZE
        ]

        batch_scores = classify_topic_batch(
            batch,
            hypothesis
        )

        scores.extend(
            batch_scores
        )

    topic_scores[topic] = np.array(scores)


assert len(topic_scores["politic"]) == len(flat_chunks)
assert len(topic_scores["religion"]) == len(flat_chunks)


# ============================================================
# REASSEMBLE TO NEWS LEVEL
# ============================================================

news_topics = []

position = 0


for chunks in all_news_chunks:

    n = len(chunks)

    politics = topic_scores["politic"][
        position:position + n
    ]

    religions = topic_scores["religion"][
        position:position + n
    ]

    position += n


    # Самый уверенный chunk по каждой теме.
    max_politics = float(
        politics.max()
    )

    max_religion = float(
        religions.max()
    )


    # Чувствительные темы только при
    # достижении установленного порога.
    candidates = []

    if max_politics >= TOPIC_THRESHOLD:
        candidates.append(
            ("politic", max_politics)
        )

    if max_religion >= TOPIC_THRESHOLD:
        candidates.append(
            ("religion", max_religion)
        )


    if not candidates:

        final_topic = "none"

    else:

        final_topic = max(
            candidates,
            key=lambda x: x[1]
        )[0]


    news_topics.append(
        final_topic
    )


assert position == len(flat_chunks)
assert len(news_topics) == len(df)


# ============================================================
# ADD ONLY THE REQUIRED COLUMN
# ============================================================

result = df.copy()

result["sensitive_topic"] = news_topics


print(
    "\nОбработано новостей:",
    len(result)
)

Новости: 70243
Chunks: 70243


Classifying politic:   0%|          | 0/1098 [00:00<?, ?it/s]

c:\Users\user\miniconda3\envs\main\Lib\site-packages\transformers\integrations\sdpa_attention.py:158: UserWarning: Using AOTriton backend for Efficient Attention forward... (Triggered internally at C:/b/pytorch/aten/src/ATen/native/transformers/hip/attention.hip:1452.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Classifying religion:   0%|          | 0/1098 [00:00<?, ?it/s]


Обработано новостей: 70243


## 5. Статистика чувствительных тем

Проверяем, сколько новостей классификатор отнёс к каждой категории.

In [15]:
topic_counts = (
    result["sensitive_topic"]
    .value_counts()
)

topic_percent = (
    result["sensitive_topic"]
    .value_counts(normalize=True)
    .mul(100)
)


topic_report = pd.DataFrame({
    "count": topic_counts,
    "percent": topic_percent.round(2)
})


print("=" * 70)
print("SENSITIVE TOPIC STATISTICS")
print("=" * 70)

print(
    topic_report.to_string()
)

print()

print(
    f"Всего новостей:    {len(result):,}"
)

print(
    f"NONE:               "
    f"{(result['sensitive_topic'] == 'none').sum():,}"
)

print(
    f"POLITIC:            "
    f"{(result['sensitive_topic'] == 'politic').sum():,}"
)

print(
    f"RELIGION:           "
    f"{(result['sensitive_topic'] == 'religion').sum():,}"
)

print()

print(
    f"Будет удалено:      "
    f"{(result['sensitive_topic'] != 'none').sum():,}"
)

print(
    f"Останется:          "
    f"{(result['sensitive_topic'] == 'none').sum():,}"
)

SENSITIVE TOPIC STATISTICS
                 count  percent
sensitive_topic                
none             69293    98.65
politic            549     0.78
religion           401     0.57

Всего новостей:    70,243
NONE:               69,293
POLITIC:            549
RELIGION:           401

Будет удалено:      950
Останется:          69,293


## 6. Быстрая ручная проверка

Перед удалением смотрим несколько примеров политики, религии и `none`.

In [16]:
for topic in ["politic", "religion", "none"]:

    print("\n" + "=" * 90)
    print("TOPIC:", topic)
    print("=" * 90)

    subset = result[
        result["sensitive_topic"] == topic
    ]

    n = min(5, len(subset))

    if n == 0:
        print("Нет примеров.")
        continue

    sample = subset.sample(
        n,
        random_state=42
    )

    for _, row in sample.iterrows():

        print(
            "\n",
            row["text"][:1200]
        )


TOPIC: politic

 "Baby Got PAC" говорит Хиллари Клинтон и демократы "приняли туда одного кандидата, который напугал Хиллари Клинтон, и этого Сена. Марко Рубио."

 Организация &lt; &lt; Юнайтед эйрлайнз &gt; &gt; сообщает 10 изменений в политике реагирования на инцидент, связанный с вывозом пассажиров

 Женщина быстро читает политические позиции кандидатов после голосования

 "При нынешних США. В рамках иммиграционной политики &quot; буквально один человек с зеленой картой &quot; может в крайнем случае привести более 270 своих родственников &quot; .

 :: программа действий в поддержку решения, предусматривающего создание двух государств;

TOPIC: religion

 Далай Лама клянется, что узнает парня на вечеринке из прошлой жизни.

 Религиозные лидеры, группы потрясены иммиграционными приказами Трампа

 В РПЦ рассказали, как привили 9 млн прихожан при помощи ГМО-ладана со «Спутником V»

 американский сикх пишет Дональду Трампу

 Ламар Одом ломает свою тишину: « Всё было моим пороком»

TOPIC: 

## 7. Фильтрация и сохранение

Удаляем только `politic` и `religion`.
Все остальные темы остаются в датасете.

In [17]:
# ============================================================
# FILTER
# ============================================================

filtered_df = result[
    result["sensitive_topic"] == "none"
].copy()


# ============================================================
# SAVE
# ============================================================

filtered_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# FINAL REPORT
# ============================================================

total = len(result)
remaining = len(filtered_df)
removed = total - remaining


print("=" * 70)
print("FINAL REPORT")
print("=" * 70)

print(
    f"Исходный full_dataset:   {total:,}"
)

print(
    f"Оставлено:               {remaining:,}"
)

print(
    f"Удалено:                 {removed:,}"
)

print(
    f"Оставлено:               "
    f"{remaining / total * 100:.2f}%"
)

print(
    f"Удалено:                 "
    f"{removed / total * 100:.2f}%"
)

print()

print(
    topic_report.to_string()
)

print()

print(
    "Saved:",
    OUTPUT_PATH.resolve()
)

FINAL REPORT
Исходный full_dataset:   70,243
Оставлено:               69,293
Удалено:                 950
Оставлено:               98.65%
Удалено:                 1.35%

                 count  percent
sensitive_topic                
none             69293    98.65
politic            549     0.78
religion           401     0.57

Saved: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\filtered_full_dataset.csv


## 8. Очистка и балансированное разделение `filtered_full_dataset.csv`

1. Загружаем физический `filtered_full_dataset.csv`.
2. Удаляем короткие строки, содержащие не более двух слов.
3. Делим оставшиеся данные 80/20.
4. Стратифицируем одновременно по `label` и `source`.
5. Сохраняем `final_train.csv` и `final_test.csv`.
6. Выводим распределение разметки и источников в процентах.

In [19]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


# ============================================================
# PATHS
# ============================================================

DATA_DIR = Path("../data/prepared_datasets")

INPUT_PATH = DATA_DIR / "filtered_full_dataset.csv"
TRAIN_PATH = DATA_DIR / "final_train.csv"
TEST_PATH = DATA_DIR / "final_test.csv"

TRAIN_RATIO = 0.80
RANDOM_STATE = 42


# ============================================================
# LOAD EXACTLY filtered_full_dataset.csv
# ============================================================

df = pd.read_csv(INPUT_PATH)

required_columns = {
    "text",
    "label",
    "source",
    "orig_lang",
    "orig_text",
    "sensitive_topic"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"В filtered_full_dataset.csv отсутствуют столбцы: "
        f"{missing_columns}"
    )

print(f"Loaded: {INPUT_PATH.resolve()}")
print(f"Original rows: {len(df):,}")


# ============================================================
# REMOVE SHORT TEXTS
# ============================================================

def count_words(text):
    """
    Считает слова, игнорируя пунктуацию,
    числа и прочие символы.
    """

    words = re.findall(
        r"[^\W\d_]+(?:[-'][^\W\d_]+)*",
        str(text),
        flags=re.UNICODE
    )

    return len(words)


word_counts = (
    df["text"]
    .fillna("")
    .astype(str)
    .map(count_words)
)


# Удаляем строки, в которых 3 слова или меньше.
short_mask = word_counts <= 3

removed_short = int(
    short_mask.sum()
)

df = df.loc[
    ~short_mask
].copy()

df.reset_index(
    drop=True,
    inplace=True
)


print("\n" + "=" * 70)
print("TEXT CLEANING")
print("=" * 70)

print(
    f"Удалено строк с <= 3 словами: "
    f"{removed_short:,}"
)

print(
    f"Осталось строк: "
    f"{len(df):,}"
)


# ============================================================
# CREATE JOINT STRATIFICATION GROUP
# label + source
# ============================================================

stratify_source = (
    df["source"]
    .fillna("__MISSING_SOURCE__")
    .astype(str)
)

df["_stratum"] = (
    df["label"].astype(str)
    + "|||"
    + stratify_source
)


# ============================================================
# TARGET TRAIN SIZE
# ============================================================

total_size = len(df)

target_train_size = round(
    total_size * TRAIN_RATIO
)


# ============================================================
# CALCULATE TRAIN COUNTS PER STRATUM
# ============================================================

group_sizes = (
    df["_stratum"]
    .value_counts()
)

ideal_train = (
    group_sizes * TRAIN_RATIO
)

train_counts = np.floor(
    ideal_train
).astype(int)


remaining = (
    target_train_size
    - train_counts.sum()
)


# Распределяем оставшиеся строки
# по группам с наибольшей дробной частью.
remainders = (
    ideal_train - train_counts
).sort_values(
    ascending=False
)


if remaining > 0:
    train_counts.loc[
        remainders.index[:remaining]
    ] += 1


# ============================================================
# SPLIT EACH STRATUM
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)

train_indices = []
test_indices = []


for stratum, group in df.groupby(
    "_stratum",
    sort=False
):

    # copy() нужен для безопасного shuffle()
    indices = (
        group.index
        .to_numpy()
        .copy()
    )

    rng.shuffle(indices)

    n_train = int(
        train_counts.loc[stratum]
    )

    train_indices.extend(
        indices[:n_train]
    )

    test_indices.extend(
        indices[n_train:]
    )


# ============================================================
# CONVERT TO NUMPY ARRAYS
# ============================================================

train_indices = np.array(
    train_indices,
    dtype=np.int64
)

test_indices = np.array(
    test_indices,
    dtype=np.int64
)


# ============================================================
# IMPORTANT SANITY CHECK
#
# Проверяем пересечение ДО reset_index().
# ============================================================

train_index_set = set(
    train_indices.tolist()
)

test_index_set = set(
    test_indices.tolist()
)

assert train_index_set.isdisjoint(
    test_index_set
), "Train и Test содержат общие исходные строки!"

assert (
    len(train_index_set)
    + len(test_index_set)
    == len(df)
), "Количество строк после split не совпадает."


# ============================================================
# SHUFFLE FINAL INDICES
# ============================================================

rng.shuffle(train_indices)
rng.shuffle(test_indices)


# ============================================================
# CREATE TRAIN / TEST
# ============================================================

train_df = df.loc[
    train_indices
].copy()

test_df = df.loc[
    test_indices
].copy()


# ============================================================
# REMOVE TECHNICAL COLUMN
# ============================================================

train_df.drop(
    columns="_stratum",
    inplace=True
)

test_df.drop(
    columns="_stratum",
    inplace=True
)


# ============================================================
# RESET OUTPUT INDICES
# ============================================================

train_df.reset_index(
    drop=True,
    inplace=True
)

test_df.reset_index(
    drop=True,
    inplace=True
)


# ============================================================
# FINAL SIZE CHECKS
# ============================================================

assert (
    len(train_df)
    + len(test_df)
    == len(df)
)

assert (
    len(train_df)
    == target_train_size
)


# ============================================================
# SAVE
# ============================================================

train_df.to_csv(
    TRAIN_PATH,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    TEST_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# FINAL SPLIT STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("FINAL SPLIT")
print("=" * 70)

print(
    f"После очистки: {len(df):,}"
)

print(
    f"Train: {len(train_df):,} "
    f"({len(train_df) / len(df) * 100:.2f}%)"
)

print(
    f"Test:  {len(test_df):,} "
    f"({len(test_df) / len(df) * 100:.2f}%)"
)


# ============================================================
# LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

train_label_pct = (
    train_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("train_%")
)

test_label_pct = (
    test_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("test_%")
)

label_report = pd.concat(
    [
        train_label_pct,
        test_label_pct
    ],
    axis=1
).fillna(0)

label_report["difference_pp"] = (
    label_report["train_%"]
    - label_report["test_%"]
)

print(
    label_report.round(2)
)


# ============================================================
# SOURCE DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("SOURCE DISTRIBUTION")
print("=" * 70)

train_source_pct = (
    train_df["source"]
    .fillna("__MISSING_SOURCE__")
    .value_counts(normalize=True)
    .mul(100)
    .rename("train_%")
)

test_source_pct = (
    test_df["source"]
    .fillna("__MISSING_SOURCE__")
    .value_counts(normalize=True)
    .mul(100)
    .rename("test_%")
)

source_report = pd.concat(
    [
        train_source_pct,
        test_source_pct
    ],
    axis=1
).fillna(0)

source_report["difference_pp"] = (
    source_report["train_%"]
    - source_report["test_%"]
)

print(
    source_report.round(2)
)


# ============================================================
# FILES
# ============================================================

print("\n" + "=" * 70)
print("SAVED FILES")
print("=" * 70)

print(
    TRAIN_PATH.resolve()
)

print(
    TEST_PATH.resolve()
)

Loaded: C:\GitHub\tg-vm-fake-detector\data\prepared_datasets\filtered_full_dataset.csv
Original rows: 69,293

TEXT CLEANING
Удалено строк с <= 3 словами: 1,370
Осталось строк: 67,923

FINAL SPLIT
После очистки: 67,923
Train: 54,338 (80.00%)
Test:  13,585 (20.00%)

LABEL DISTRIBUTION
       train_%  test_%  difference_pp
label                                
0        56.57   56.58           -0.0
1        43.43   43.42            0.0

SOURCE DISTRIBUTION
                 train_%  test_%  difference_pp
source                                         
onion_huffpost     40.21   40.21           0.00
gossipcop          14.13   14.13          -0.00
pubhealth_train    13.52   13.53          -0.01
nplus1             11.20   11.20           0.00
danya_train         8.37    8.37           0.00
train_stances       6.40    6.40          -0.00
pubhealth_test      1.69    1.69           0.00
pubhealth_dev       1.68    1.68           0.00
test_stances        1.60    1.60           0.00
panorama       